In [ ]:
import os
import numpy as np
import pandas as pd

# ============================================================
# LOAD FROZEN PRODUCTION MILP ENGINE
# ============================================================

%run milp_engine.ipynb


# ============================================================
# FOCUSED DURATION-CAP SENSITIVITY CONFIGURATION
# ============================================================

MODELS = {
    "LightGBM": "Prescriptive/LightGBM_prescriptive_optimizer_inputs.csv",
    "RF": "Prescriptive/RF_prescriptive_optimizer_inputs.csv",
    "XGBoost": "Prescriptive/XGBoost_prescriptive_optimizer_inputs.csv",
}

COHORT_PATH = "Cohort/fixed_cohort.csv"
PRODUCTION_SUMMARY = "Results/all_models_summary.csv"
OUTPUT_DIR = "Results"

LAMBDA = 1.0

N_CASES = 12
N_ROOMS = 3
ROOM_CAPACITY = 480
TURNOVER = 20
BALANCE_WEIGHT = 0.10
TIME_LIMIT = 300

MAIN_CAP = 360.0
SENSITIVITY_CAP = None

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("=" * 70)
print("FOCUSED DURATION-CAP SENSITIVITY")
print("=" * 70)
print("Models              :", list(MODELS))
print("Lambda              :", LAMBDA)
print("Main cap            :", MAIN_CAP)
print("Sensitivity cap     :", SENSITIVITY_CAP)
print("New MILP runs       :", len(MODELS))
print("Time limit per run  :", TIME_LIMIT, "s")

In [ ]:
# ============================================================
# LOAD AND VERIFY FROZEN COHORT
# ============================================================

assert os.path.exists(COHORT_PATH), f"Missing cohort: {COHORT_PATH}"
assert os.path.exists(PRODUCTION_SUMMARY), (
    f"Missing production summary: {PRODUCTION_SUMMARY}"
)

cohort = pd.read_csv(COHORT_PATH)

assert "LOG_ID" in cohort.columns
assert len(cohort) == N_CASES
assert cohort["LOG_ID"].is_unique

frozen_ids = cohort["LOG_ID"].astype(str).tolist()


# ============================================================
# LOAD PRODUCTION RESULTS
# ============================================================

production = pd.read_csv(PRODUCTION_SUMMARY)

required_prod_cols = {
    "Model", "Lambda", "Objective",
    "Gap", "Solve_Time", "N_Capped"
}

assert required_prod_cols.issubset(production.columns), (
    f"Production summary missing columns: "
    f"{required_prod_cols - set(production.columns)}"
)


# ============================================================
# VERIFY MODEL INPUTS + EXACT FROZEN COHORT
# ============================================================

verified = {}

for model_name, path in MODELS.items():

    print("\n" + "-" * 70)
    print(f"Verifying {model_name}")
    print("-" * 70)

    assert os.path.exists(path), f"Missing input: {path}"

    df = pd.read_csv(path)

    required = {
        "LOG_ID",
        "DURATION_P50_MINS",
        "DURATION_P90_MINS"
    }

    assert required.issubset(df.columns), (
        f"{model_name}: missing "
        f"{required - set(df.columns)}"
    )

    assert df["LOG_ID"].is_unique

    df = df.copy()
    df["LOG_ID"] = df["LOG_ID"].astype(str)

    missing = set(frozen_ids) - set(df["LOG_ID"])
    assert not missing, f"{model_name}: missing IDs {missing}"

    day = (
        df.set_index("LOG_ID")
          .loc[frozen_ids]
          .reset_index()
    )

    assert day["LOG_ID"].tolist() == frozen_ids
    assert len(day) == N_CASES

    # Prediction sanity
    assert np.isfinite(
        day["DURATION_P50_MINS"]
    ).all()

    assert np.isfinite(
        day["DURATION_P90_MINS"]
    ).all()

    assert (
        day["DURATION_P50_MINS"] > 0
    ).all()

    assert (
        day["DURATION_P90_MINS"]
        >= day["DURATION_P50_MINS"]
    ).all()

    verified[model_name] = day

    # λ=1 means planning duration = P90
    p90 = day["DURATION_P90_MINS"].to_numpy(float)
    affected = p90 > MAIN_CAP

    print(
        f"{model_name}: PASS | "
        f"full={df.shape} | fixed={day.shape}"
    )

    print(
        f"Cases affected by 360-min cap at lambda=1: "
        f"{affected.sum()}/{N_CASES}"
    )

    print(
        f"Maximum uncapped P90: {p90.max():.3f} min"
    )

print("\n" + "=" * 70)
print("PRE-RUN INTEGRITY CHECK: PASS")
print("=" * 70)

In [ ]:
# ============================================================
# RUN THREE UNCAPPED λ=1 SENSITIVITY EXPERIMENTS
# ============================================================

uncapped_records = []

for model_name, df_day in verified.items():

    print("\n" + "=" * 70)
    print(
        f"UNCAPPED SENSITIVITY | "
        f"{model_name} | lambda={LAMBDA}"
    )
    print("=" * 70)

    result = solve_or_milp(
        df_day=df_day,
        lam=LAMBDA,
        n_rooms=N_ROOMS,
        room_capacity=ROOM_CAPACITY,
        turnover=TURNOVER,
        balance_weight=BALANCE_WEIGHT,
        duration_cap=SENSITIVITY_CAP,
        time_limit=TIME_LIMIT,
    )

    assert result is not None, (
        f"{model_name}: no feasible incumbent returned."
    )

    record = {
        "Model": model_name,
        "Lambda": LAMBDA,
        "Cap_Setting": "Uncapped",
        "Duration_Cap": np.nan,

        "Objective": result.get("Objective", np.nan),
        "Overtime": result.get("Overtime", np.nan),
        "Idle": result.get("Idle", np.nan),
        "Makespan": result.get("Makespan", np.nan),

        "Sched_Util": result.get("Sched_Util", np.nan),
        "Cap_Util": result.get("Cap_Util", np.nan),
        "Buffer_Ratio": result.get("Buffer_Ratio", np.nan),
        "Cap_Expansion": result.get("Cap_Expansion", np.nan),
        "OnTime_Rate": result.get("OnTime_Rate", np.nan),
        "Load_Gap": result.get("Load_Gap", np.nan),

        "N_Capped": result.get("N_Capped", np.nan),
        "Pct_Capped": result.get("Pct_Capped", np.nan),
        "Max_Uncapped_Duration":
            result.get("Max_Uncapped_Duration", np.nan),
        "Max_Planning_Duration":
            result.get("Max_Planning_Duration", np.nan),

        "Solve_Time": result.get("Solve_Time", np.nan),
        "Variables": result.get("Variables", np.nan),
        "Constraints": result.get("Constraints", np.nan),
        "Nodes": result.get("Nodes", np.nan),
        "Gap": result.get("Gap", np.nan),
        "Status": result.get("Status", np.nan),

        "Reached_1pct_Gap":
            result.get("Reached_1pct_Gap", False),
        "Hit_Time_Limit":
            result.get("Hit_Time_Limit", False),
    }

    uncapped_records.append(record)

    # Checkpoint after every run
    uncapped_df = pd.DataFrame(uncapped_records)

    uncapped_df.to_csv(
        os.path.join(
            OUTPUT_DIR,
            "duration_cap_sensitivity_uncapped.csv"
        ),
        index=False
    )

    print(
        f"Objective       : "
        f"{record['Objective']:.4f}"
    )

    print(
        f"Overtime        : "
        f"{record['Overtime']:.4f}"
    )

    print(
        f"Makespan        : "
        f"{record['Makespan']:.4f}"
    )

    print(
        f"Gap             : "
        f"{record['Gap']:.4%}"
    )

    print(
        f"Solve time      : "
        f"{record['Solve_Time']:.2f} s"
    )

    print(
        f"Hit time limit  : "
        f"{record['Hit_Time_Limit']}"
    )

    print(
        f"Max duration    : "
        f"{record['Max_Planning_Duration']:.3f} min"
    )

print("\n" + "=" * 70)
print("UNCAPPED SENSITIVITY RUNS FINISHED")
print("=" * 70)
print("Completed:", len(uncapped_records), "/ 3")

In [ ]:
# ============================================================
# BUILD CAPPED vs UNCAPPED COMPARISON
# ============================================================

uncapped_df = pd.DataFrame(uncapped_records)

comparison_rows = []

for model_name in MODELS:

    # --------------------------------------------------------
    # CAPPED BASELINE FROM FROZEN 33-RUN PRODUCTION EXPERIMENT
    # --------------------------------------------------------

    capped_match = production[
        (production["Model"] == model_name)
        &
        np.isclose(
            production["Lambda"].astype(float),
            LAMBDA
        )
    ]

    assert len(capped_match) == 1, (
        f"{model_name}: expected exactly one "
        f"production lambda=1 row, found "
        f"{len(capped_match)}"
    )

    capped = capped_match.iloc[0]

    # --------------------------------------------------------
    # NEW UNCAPPED RESULT
    # --------------------------------------------------------

    uncapped_match = uncapped_df[
        uncapped_df["Model"] == model_name
    ]

    assert len(uncapped_match) == 1

    uncapped = uncapped_match.iloc[0]

    row = {
        "Model": model_name,
        "Lambda": LAMBDA,

        "N_Capped_Main":
            capped.get("N_Capped", np.nan),

        "Capped_Objective":
            capped.get("Objective", np.nan),

        "Uncapped_Objective":
            uncapped["Objective"],

        "Delta_Objective":
            uncapped["Objective"]
            - capped.get("Objective", np.nan),

        "Pct_Delta_Objective":
            (
                (
                    uncapped["Objective"]
                    - capped.get("Objective", np.nan)
                )
                / capped.get("Objective", np.nan)
                * 100
            ),

        "Capped_Overtime":
            capped.get("Overtime", np.nan),

        "Uncapped_Overtime":
            uncapped["Overtime"],

        "Delta_Overtime":
            uncapped["Overtime"]
            - capped.get("Overtime", np.nan),

        "Capped_Makespan":
            capped.get("Makespan", np.nan),

        "Uncapped_Makespan":
            uncapped["Makespan"],

        "Delta_Makespan":
            uncapped["Makespan"]
            - capped.get("Makespan", np.nan),

        "Capped_Load_Gap":
            capped.get("Load_Gap", np.nan),

        "Uncapped_Load_Gap":
            uncapped["Load_Gap"],

        "Capped_Gap":
            capped.get("Gap", np.nan),

        "Uncapped_Gap":
            uncapped["Gap"],

        "Capped_Solve_Time":
            capped.get("Solve_Time", np.nan),

        "Uncapped_Solve_Time":
            uncapped["Solve_Time"],

        "Uncapped_Max_Planning_Duration":
            uncapped["Max_Planning_Duration"],
    }

    comparison_rows.append(row)


comparison_df = pd.DataFrame(comparison_rows)

comparison_path = os.path.join(
    OUTPUT_DIR,
    "duration_cap_sensitivity_comparison.csv"
)

comparison_df.to_csv(
    comparison_path,
    index=False
)


# ============================================================
# FINAL OUTPUT
# ============================================================

print("\n" + "=" * 90)
print("λ=1 FOCUSED DURATION-CAP SENSITIVITY — FINAL COMPARISON")
print("=" * 90)

display_cols = [
    "Model",
    "N_Capped_Main",
    "Capped_Objective",
    "Uncapped_Objective",
    "Delta_Objective",
    "Pct_Delta_Objective",
    "Capped_Overtime",
    "Uncapped_Overtime",
    "Delta_Overtime",
    "Capped_Makespan",
    "Uncapped_Makespan",
    "Delta_Makespan",
]

print(
    comparison_df[
        display_cols
    ].to_string(index=False)
)


# ============================================================
# RF NEGATIVE-CONTROL CHECK
# ============================================================

rf = comparison_df[
    comparison_df["Model"] == "RF"
].iloc[0]

print("\n" + "-" * 90)
print("RF NEGATIVE-CONTROL CHECK")
print("-" * 90)

print(
    f"N capped in main experiment : "
    f"{rf['N_Capped_Main']}"
)

print(
    f"Objective difference        : "
    f"{rf['Delta_Objective']:.6f}"
)

print(
    f"Overtime difference         : "
    f"{rf['Delta_Overtime']:.6f}"
)

print(
    f"Makespan difference         : "
    f"{rf['Delta_Makespan']:.6f}"
)

if rf["N_Capped_Main"] == 0:
    print(
        "\nExpected interpretation: RF is the negative control "
        "because its lambda=1 planning durations are unaffected "
        "by the 360-min cap."
    )


print("\nSaved:")
print(
    " - Results/"
    "duration_cap_sensitivity_uncapped.csv"
)
print(
    " - Results/"
    "duration_cap_sensitivity_comparison.csv"
)